# SQL Server → Python → Visualization Pipeline

**Author:** Khaylub Thompson-Calvin · Portfolio project from PCC CIS277A (Data Analytics) coursework

This notebook demonstrates an end-to-end data pipeline: query a SQL Server database with `pyodbc`, load
results into a pandas DataFrame, reshape with `pivot`, and visualize with matplotlib.

**Data:** U.S. Social Security Administration baby-names dataset (public data), hosted on a course database
server during the class.

> **How to read this notebook.** The course database requires student credentials and is not publicly
> reachable, so **this notebook is preserved as annotated code and is not executed here** — the cells
> below have no saved outputs. The resulting chart is included separately in `screenshots/`.
> For a version you can actually run end to end, see
> [`public-ssa-analysis.ipynb`](public-ssa-analysis.ipynb), which reproduces the same analysis from the
> public SSA files with no database access required.

**Note on credentials:** connection settings are read from environment variables. Never hard-code server
names, usernames, or passwords in a notebook.


In [ ]:
import os

import pyodbc
import pandas as pd
import matplotlib.pyplot as plt

# Connection settings come from environment variables (never hard-coded).
# The ODBC driver name is configurable too: driver versions differ by machine,
# so hard-coding one makes the notebook fail on a perfectly good setup.
# Driver 18 defaults to encryption, which is why TrustServerCertificate is exposed here.
driver = os.environ.get("DB_DRIVER", "ODBC Driver 18 for SQL Server")
trust_cert = os.environ.get("DB_TRUST_SERVER_CERT", "yes")

connection = pyodbc.connect(
    f"DRIVER={{{driver}}};"
    f"SERVER={os.environ['DB_SERVER']};"
    f"DATABASE={os.environ['DB_NAME']};"
    f"UID={os.environ['DB_USER']};"
    f"PWD={os.environ['DB_PASSWORD']};"
    f"Encrypt=yes;"
    f"TrustServerCertificate={trust_cert};"
)


## Query: two name spellings over time

Pull the yearly counts for the names *Marc* and *Mark* (male) so we can compare how the two spellings trended across a century.

In [ ]:
df = pd.read_sql(
    """
    SELECT Name, Gender, Year, NameCount
    FROM all_data
    WHERE (Name = 'Marc' OR Name = 'Mark')
      AND Gender = 'M'
    ORDER BY Year
    """,
    connection,
)
df.head()

## Reshape and plot

Pivot the long-format result (one row per name per year) into one column per name, indexed by year, then plot both series.

In [ ]:
name_data = df.pivot(index="Year", columns="Name", values="NameCount")
name_data.plot()
plt.ylabel("Babies named per year")
plt.title("Marc vs. Mark, U.S. male births by year")
plt.show()

### Result

The chart below is the output this pipeline produced when it was run against the course database.
It is included as an image because this notebook is not executed here (see the note at the top).

![Marc vs Mark trend](../screenshots/names-trend-chart.png)

### What this demonstrates
- Connecting Python to SQL Server (`pyodbc` + Microsoft ODBC Driver, driver name configurable)
- Writing a filtered, ordered SQL query from Python
- Loading query results into pandas and reshaping with `pivot`
- Producing a readable time-series comparison with matplotlib
- Keeping credentials out of source with environment variables
